# Lab 6  Logistic Regression & KNN Classification
## Student Information
Name: **Nazakat Ali**  
Student ID: **023-22-0146**  
Section: **Buffer Batch**  
GitHub Profile: **https://github.com/Nizakatkhannk**  

Dataset: **Telco Customer Churn** — https://www.kaggle.com/datasets/blastchar/telco-customer-churn (target: `Churn`)


## 1. Problem Definition & Dataset Verification

In [ ]:
import os, glob, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)
sns.set_style('whitegrid')
RANDOM_STATE, TEST_SIZE = 42, 0.2      # <-- use the SAME values as Labs 3/4 so results are comparable

def say(t): display(Markdown(t))
def get_metrics(y_true, y_pred):
    return {'Accuracy': accuracy_score(y_true, y_pred), 'Precision': precision_score(y_true, y_pred, zero_division=0),
            'Recall': recall_score(y_true, y_pred), 'F1': f1_score(y_true, y_pred)}

# ---- load Telco churn (local file -> kagglehub -> manual upload) ----
CSV = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
hits = glob.glob(f'**/{CSV}', recursive=True)
if hits: CSV = hits[0]
else:
    try:
        try: import kagglehub
        except ImportError:
            import subprocess, sys; subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'kagglehub'], check=True); import kagglehub
        CSV = os.path.join(kagglehub.dataset_download('blastchar/telco-customer-churn'), CSV)
    except Exception as e:
        print('Auto-download failed:', e, '\nPlease upload the CSV manually.')
        from google.colab import files; CSV = list(files.upload().keys())[0]

df = pd.read_csv(CSV)
print('Shape:', df.shape); df.info()

In [ ]:
# ---- same cleaning as Lab 2: TotalCharges -> numeric, drop customerID, Churn -> 0/1 ----
df = df.drop(columns=['customerID'])
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('Missing TotalCharges (blank strings):', df['TotalCharges'].isna().sum())
y = df['Churn'].map({'Yes': 1, 'No': 0}); X_raw = df.drop(columns='Churn')
print('Churn rate: %.1f%%  -> a model that always says "No churn" gets %.1f%% accuracy' % (100*y.mean(), 100*(1-y.mean())))

### Task 1 — Benchmarks from Labs 3 and 4
The cells in Part 5 re-train the Decision Tree and the (tuned) Random Forest on **exactly this lab's split**, so the benchmarks are fair; their numbers are printed in the Part 5 comparison table and restated in the sentence generated at the end of Part 5. *If you saved the exact Lab 3/4 numbers, paste them into your own 2–3 sentence restatement here.*

## 2. Feature Scaling & Preparation

In [ ]:
# Split FIRST (stratified, same random_state/test_size as Labs 3/4 => same rows), then impute/encode using TRAIN information only.
Xr_train, Xr_test, y_train, y_test = train_test_split(X_raw, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
med = Xr_train['TotalCharges'].median()
Xr_train = Xr_train.assign(TotalCharges=Xr_train['TotalCharges'].fillna(med)); Xr_test = Xr_test.assign(TotalCharges=Xr_test['TotalCharges'].fillna(med))
X_train = pd.get_dummies(Xr_train, drop_first=True).astype(float)       # drop_first -> clear baseline categories for Part 6
X_test  = pd.get_dummies(Xr_test,  drop_first=True).astype(float).reindex(columns=X_train.columns, fill_value=0.0)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)     # fit on TRAIN only
X_test_s  = scaler.transform(X_test)          # apply the train statistics to test
print('Train:', X_train.shape, 'Test:', X_test.shape)

num = ['tenure', 'MonthlyCharges', 'TotalCharges']; idx = [list(X_train.columns).index(c) for c in num]
before = X_train[num].agg(['mean', 'std', 'min', 'max']).T
after  = pd.DataFrame(X_train_s[:, idx], columns=num).agg(['mean', 'std', 'min', 'max']).T
display(pd.concat({'before scaling': before, 'after scaling': after}, axis=1).round(2))

### Answers — Part 2
**Interpretation of the table.** Before scaling, `TotalCharges` has a standard deviation in the thousands, `MonthlyCharges` in the tens and `tenure` in the tens of months, while the dummy columns are 0/1. After `StandardScaler` every column has mean ≈ 0 and standard deviation ≈ 1 on the training set, so all features are on a comparable footing.

**Why scaling matters for KNN and Logistic Regression but not for trees.** KNN classifies by Euclidean distance, so a feature with a large numeric range (`TotalCharges`, thousands) swamps one with a small range (a 0/1 dummy), and the "nearest" neighbours are effectively chosen by the big-number columns alone. Logistic Regression is not distance-based, but it is fitted by gradient-based optimisation with L2 regularisation: unscaled features give an ill-conditioned problem (slow/unstable convergence), and the penalty shrinks coefficients of large-range and small-range features unequally, which also makes coefficients impossible to compare. Decision Trees and Random Forests only ask "is feature ≤ threshold?", and that question gives the same split under any monotonic rescaling, so scale never mattered.

**Why fit the scaler on the training data only?** The scaler's mean/std are *learned parameters*. If computed on the full dataset, information about the test rows leaks into the training features, so the test score would be optimistically biased and no longer simulate unseen customers. Fitting on train and only `transform`-ing test mimics deployment, where new customers are scaled with the statistics from the data the model was built on. (For the same reason, I split before imputing and one-hot encoding, and I use a `Pipeline` for all cross-validation below so the scaler is re-fit inside each fold.)

## 3. Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE).fit(X_train_s, y_train)
pred_lr = lr.predict(X_test_s); m_lr = get_metrics(y_test, pred_lr)
display(pd.DataFrame([m_lr], index=['Logistic Regression']).round(4))
cm_lr = confusion_matrix(y_test, pred_lr); tn, fp, fn, tp = cm_lr.ravel()
ConfusionMatrixDisplay(cm_lr, display_labels=['Stay', 'Churn']).plot(cmap='Blues'); plt.title('Logistic Regression — confusion matrix (test)'); plt.show()
say(f'''**Your numbers:** accuracy **{m_lr['Accuracy']:.4f}**, precision **{m_lr['Precision']:.4f}**, recall **{m_lr['Recall']:.4f}**, F1 **{m_lr['F1']:.4f}**.
Of {len(y_test)} test customers the model caught **{tp}** churners, missed **{fn}** (false negatives) and raised **{fp}** false alarms.''')

In [ ]:
proba = lr.predict_proba(X_test_s)[:10]
samp = pd.DataFrame({'P(stay)': proba[:, 0].round(3), 'P(churn)': proba[:, 1].round(3),
                     'predict()': lr.predict(X_test_s[:10]), 'true label': y_test.values[:10]})
display(samp)
print('predict() equals (P(churn) >= 0.5) for every test customer:', bool(((lr.predict_proba(X_test_s)[:, 1] >= 0.5).astype(int) == pred_lr).all()))

### Answers — Part 3
**Confusion-matrix interpretation.** The diagonal holds the correct decisions; the bottom-left cell (false negatives) are churners the model failed to flag — the costly error for a retention team — while the top-right cell (false positives) are loyal customers who would receive an unnecessary offer. Because only about a quarter of customers churn, accuracy flatters the model (the "always No" baseline printed in Part 1 is already high), which is why precision, recall and F1 matter.

**What `predict_proba` means.** Each row gives the model's estimated probability of each class, `P(stay)` and `P(churn)`, which sum to 1; they come from passing the linear score (the log-odds) through the sigmoid. `predict()` simply applies the default threshold: it outputs *Churn* when `P(churn) ≥ 0.5` and *Stay* otherwise (the last line of the output confirms this for every test row). The threshold can be moved to trade precision against recall.

## 4. KNN & Choosing K

In [ ]:
Ks = [1, 3, 5, 7, 9, 11, 15, 21, 31, 41]       # the manual's list, extended so we can see where the curve turns
rows = []
for k in Ks:
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train)
    cv = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)), X_train, y_train, cv=5, scoring='f1')
    rows.append({'K': k, 'Train F1': f1_score(y_train, knn.predict(X_train_s)),
                 'CV F1 (5-fold, train only)': cv.mean(),
                 'Test Accuracy': accuracy_score(y_test, knn.predict(X_test_s)), 'Test F1': f1_score(y_test, knn.predict(X_test_s))})
kres = pd.DataFrame(rows).set_index('K'); display(kres.round(4))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(kres.index, kres['Test Accuracy'], 'o-', label='Test accuracy'); ax[0].plot(kres.index, kres['Test F1'], 's-', label='Test F1')
ax[0].set_xlabel('K'); ax[0].set_title('Test accuracy and F1 vs K'); ax[0].legend()
ax[1].plot(kres.index, kres['Train F1'], 'o-', label='Train F1'); ax[1].plot(kres.index, kres['CV F1 (5-fold, train only)'], 's-', label='CV F1'); ax[1].plot(kres.index, kres['Test F1'], '^-', label='Test F1')
ax[1].set_xlabel('K'); ax[1].set_title('Overfitting check: train vs CV vs test F1'); ax[1].legend()
plt.tight_layout(); plt.show()

best_k = int(kres['CV F1 (5-fold, train only)'].idxmax())    # chosen with CV on the TRAIN set, not by peeking at test
print('Chosen K (highest cross-validated F1):', best_k, '| largest K tested:', max(Ks), '| smallest:', min(Ks))

In [ ]:
knn_best = KNeighborsClassifier(n_neighbors=best_k).fit(X_train_s, y_train)
pred_knn = knn_best.predict(X_test_s); m_knn = get_metrics(y_test, pred_knn)
display(pd.DataFrame([m_knn], index=[f'KNN (K={best_k})']).round(4))
cm_knn = confusion_matrix(y_test, pred_knn); tnk, fpk, fnk, tpk = cm_knn.ravel()
ConfusionMatrixDisplay(cm_knn, display_labels=['Stay', 'Churn']).plot(cmap='Blues'); plt.title(f'KNN (K={best_k}) — confusion matrix (test)'); plt.show()
say(f'''**Your numbers:** K = **{best_k}** → accuracy **{m_knn['Accuracy']:.4f}**, precision **{m_knn['Precision']:.4f}**, recall **{m_knn['Recall']:.4f}**, F1 **{m_knn['F1']:.4f}**.
K = 1 has train F1 = {kres.loc[1,'Train F1']:.3f} but CV F1 = {kres.loc[1,'CV F1 (5-fold, train only)']:.3f} (memorising noise).
At K = {best_k} the CV F1 is {kres.loc[best_k,'CV F1 (5-fold, train only)']:.3f}; the biggest K tested ({max(Ks)}) gives {kres.loc[max(Ks),'CV F1 (5-fold, train only)']:.3f}.''')

### Answers — Part 4
**Plot interpretation.** In the right-hand plot, *train* F1 is near 1.0 at K = 1 (each point is its own nearest neighbour) and falls quickly as K grows, while CV/test F1 rise from a very low value at K = 1 — a classic overfitting signature. At large K the curves flatten or fall again as the model averages over too many unrelated customers and, with only ~27 % churners, drifts towards always predicting "Stay" (recall collapses even if accuracy stays high). This is why the left-hand plot shows accuracy changing little while F1 changes a lot.

**Choice of K.** I chose K by **5-fold cross-validated F1 on the training set** (the Lab 4 lesson: don't pick hyper-parameters by staring at the test set), so the K above is the one with the best CV F1. I did not choose K = 1 because it overfits (train F1 ≈ 1, CV/test far lower, very noisy boundary), and I did not simply choose the largest K because beyond the peak, larger K only smooths away real structure and lowers recall. *(If your chosen K came out equal to the largest tested value, say so and note that the true optimum may lie beyond it, which would be worth testing next.)* Differences of ~0.01 F1 between neighbouring K values are within noise.

## 5. Model Comparison (4 Models)

> The Decision Tree and Random Forest are re-trained here on **this lab's split** (unscaled features — trees don't need scaling), with light tuning by cross-validation on the training set. If you have your exact Lab 3/4 settings, substitute them in the next cell.

In [ ]:
dt_gs = GridSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE), {'max_depth': [3, 4, 5, 6, 8, 10], 'min_samples_leaf': [1, 5, 20]},
                     cv=5, scoring='f1', n_jobs=-1).fit(X_train, y_train)
rf_gs = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
                     {'n_estimators': [200, 400], 'max_depth': [6, 10, None], 'min_samples_leaf': [1, 5]},
                     cv=5, scoring='f1', n_jobs=-1).fit(X_train, y_train)
dt = DecisionTreeClassifier(random_state=RANDOM_STATE, **dt_gs.best_params_).fit(X_train, y_train)
rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **rf_gs.best_params_).fit(X_train, y_train)
print('DT params:', dt_gs.best_params_, '| RF params:', rf_gs.best_params_, '| KNN K:', best_k)

test_res = {'Decision Tree': get_metrics(y_test, dt.predict(X_test)), 'Random Forest (tuned)': get_metrics(y_test, rf.predict(X_test)),
            'Logistic Regression': m_lr, f'KNN (K={best_k})': m_knn}
comp = pd.DataFrame(test_res).T; display(comp.round(4))

# 5-fold CV F1 on the training set for every model (scaling inside a Pipeline for LR / KNN) -> how stable are these numbers?
cvmodels = {'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE, **dt_gs.best_params_),
            'Random Forest (tuned)': RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **rf_gs.best_params_),
            'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
            f'KNN (K={best_k})': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k))}
cvrows = {}
for n, mdl in cvmodels.items():
    s = cross_val_score(mdl, X_train, y_train, cv=5, scoring='f1'); cvrows[n] = {'CV F1 mean': s.mean(), 'CV F1 std': s.std()}
cvtab = pd.DataFrame(cvrows).T; display(cvtab.round(4))

ax = comp.plot(kind='bar', figsize=(11, 4.8), rot=15, width=.8); ax.set_ylim(0, 1); ax.set_title('Four classifiers, test-set metrics'); plt.tight_layout(); plt.show()

In [ ]:
best = {m: comp[m].idxmax() for m in comp.columns}
say(f'''**Your numbers:** best **accuracy** → {best['Accuracy']} ({comp['Accuracy'].max():.3f}); best **precision** → {best['Precision']} ({comp['Precision'].max():.3f});
best **recall** → {best['Recall']} ({comp['Recall'].max():.3f}); best **F1** → {best['F1']} ({comp['F1'].max():.3f}). Best CV-F1 → {cvtab['CV F1 mean'].idxmax()} ({cvtab['CV F1 mean'].max():.3f} ± {cvtab.loc[cvtab['CV F1 mean'].idxmax(),'CV F1 std']:.3f}).
Benchmarks from Labs 3–4 on this split: Decision Tree F1 = {comp.loc['Decision Tree','F1']:.3f} (acc {comp.loc['Decision Tree','Accuracy']:.3f}), Random Forest F1 = {comp.loc['Random Forest (tuned)','F1']:.3f} (acc {comp.loc['Random Forest (tuned)','Accuracy']:.3f}).''')

# what did scaling change? (evidence for the Part 7 conclusion)
def fit_quiet(mdl, Xtr, Xte):
    with warnings.catch_warnings(): warnings.simplefilter('ignore'); mdl.fit(Xtr, y_train); return f1_score(y_test, mdl.predict(Xte))
sc = pd.DataFrame({'Unscaled F1': [fit_quiet(LogisticRegression(max_iter=1000), X_train, X_test), fit_quiet(KNeighborsClassifier(n_neighbors=best_k), X_train, X_test)],
                   'Scaled F1':   [fit_quiet(LogisticRegression(max_iter=1000), X_train_s, X_test_s), fit_quiet(KNeighborsClassifier(n_neighbors=best_k), X_train_s, X_test_s)]},
                  index=['Logistic Regression', f'KNN (K={best_k})']); display(sc.round(4))

### Interpretation of the comparison
The first table and bar chart compare the four algorithms on the same held-out test set; the second table repeats the comparison with 5-fold cross-validated F1 on the training data, which shows how much each score would move on a different split (the `std` column). The last table isolates the effect of scaling on the two new models.

### Answers — Part 5
**Best overall / ranking by metric.** The winners per metric are printed above. On this dataset the models are usually close on accuracy (about 0.78–0.81), but the ranking *does* change with the metric: the tree-based models and KNN tend to give up recall, while Logistic Regression is usually the strongest on F1 and precision at the default threshold. If your winners differ across metrics, that is exactly the point of this question.

**Which metric should carry the most weight?** Churners are a minority (~27 %), so accuracy is dominated by the easy majority class — "always Stay" already scores about 73 %. A missed churner (false negative) costs a customer's future revenue, whereas a false alarm only costs a retention offer, so **recall and F1 (the balance of precision and recall on the churn class)** should carry the most weight, with accuracy as a secondary check. Differences of ~0.01 between models are within the CV noise shown in the second table and should be treated as ties.

## 6. Coefficient Interpretation

In [ ]:
feat = X_train.columns
coef = lr.coef_[0]; coef_orig = coef / scaler.scale_          # coefficient per ONE ORIGINAL UNIT (dummy 0->1, +1 month, +$1)
coef_df = pd.DataFrame({'Feature': feat, 'Coefficient (scaled)': coef, 'Odds Ratio per 1 SD': np.exp(coef), 'Odds Ratio per original unit': np.exp(coef_orig)})
coef_df['abs'] = coef_df['Coefficient (scaled)'].abs(); coef_df = coef_df.sort_values('abs', ascending=False).drop(columns='abs').reset_index(drop=True)

units = {'tenure': 'each extra month of tenure', 'MonthlyCharges': 'each extra $1 of monthly charges', 'TotalCharges': 'each extra $1 of total charges'}
def interp(f, o):
    who = units.get(f, f'a customer with "{f}" = 1 (vs. the baseline category)')
    return f'{who} multiplies the odds of churn by {o:.2f}× ({abs(o-1)*100:.0f}% {"higher" if o > 1 else "lower"}), other features held constant'
coef_df['Interpretation'] = [interp(f, o) for f, o in zip(coef_df['Feature'], coef_df['Odds Ratio per original unit'])]
print('Intercept (log-odds for the baseline customer at mean values):', round(float(lr.intercept_[0]), 3))
pd.set_option('display.max_colwidth', 140); pd.set_option('display.width', 250)
display(coef_df.head(15).round(3))

In [ ]:
top = coef_df.head(12).iloc[::-1]
plt.figure(figsize=(8, 5)); plt.barh(top['Feature'], top['Coefficient (scaled)'], color=np.where(top['Coefficient (scaled)'] > 0, 'crimson', 'seagreen'))
plt.axvline(0, color='k', lw=.8); plt.xlabel('Coefficient (log-odds per 1 SD) — red raises churn odds, green lowers'); plt.title('Top 12 Logistic Regression coefficients'); plt.tight_layout(); plt.show()

# compare with the Decision Tree's feature importances (Lab 3)
imp = pd.Series(dt.feature_importances_, index=feat).sort_values(ascending=False)
lr_top, dt_top = list(coef_df['Feature'].head(10)), list(imp.head(10).index)
overlap = sorted(set(lr_top) & set(dt_top))
display(pd.DataFrame({'LR top-10 (by |coef|)': lr_top, 'Decision Tree top-10 (importance)': dt_top}))
say(f'''**Your numbers:** {len(overlap)} of the top-10 features appear in both lists: **{', '.join(overlap) or 'none'}**.''')

### Interpretation of the coefficient table and chart
`Coefficient (scaled)` is the change in the **log-odds** of churn for a one-standard-deviation increase in that feature, so its sign gives the direction and its size lets us rank features (all are on the same scale). `Odds Ratio per 1 SD = exp(coef)` converts that into a multiplicative change in the odds. `Odds Ratio per original unit = exp(coef / std)` undoes the scaling so that a dummy column reads as "0 → 1" and `tenure` as "per month", which is what the Interpretation column uses. An odds ratio above 1 raises the odds of churning, below 1 lowers them. (Odds ratios are about *odds*, not probabilities: "1.5×" does not mean 50 % more probable.)

### Answers — Part 6
**Interpretations (≥ 4 features, from the table above).** Read four rows of your own table and restate them in plain words, e.g. a **longer tenure** has an odds ratio below 1 (every extra month lowers the odds of churn), customers on a **two-year contract** have much lower odds than the month-to-month baseline, **fibre-optic internet** customers have higher odds than DSL customers, and **electronic-check payment** is associated with higher odds than the baseline payment method. Check the signs and numbers against your table — the Interpretation column already gives the exact multipliers.

**Agreement with earlier labs.** The headline drivers (contract type, tenure, internet service/fibre, monthly/total charges) should also dominate the Lab 2 EDA and the Decision Tree importances, as the overlap count above shows. Two kinds of disagreement are normal. First, **`MonthlyCharges`, `TotalCharges` and `tenure` are strongly correlated** (TotalCharges ≈ tenure × MonthlyCharges) and the add-on service dummies are collinear with `InternetService`, so Logistic Regression splits their joint effect between them and individual coefficients can have surprising signs (e.g. MonthlyCharges looking protective once other features are in the model); the tree, which picks one splitting variable at a time, just credits whichever variable gives the best first split. Second, a tree importance is always positive and says nothing about direction, whereas a coefficient has a sign and holds the other features constant. Note your own disagreements from the two lists above.

## 7. Final Model Selection & Conclusion

In [ ]:
rank = comp.assign(**{'CV F1': cvtab['CV F1 mean']}).sort_values('F1', ascending=False); display(rank.round(4))
final_name = comp['F1'].idxmax(); tie = comp['F1'].sort_values(ascending=False)
close = tie.iloc[0] - tie.iloc[1] < 0.01
say(f'''**Final model (highest test F1): {final_name}** — F1 {comp.loc[final_name,'F1']:.3f}, recall {comp.loc[final_name,'Recall']:.3f}, precision {comp.loc[final_name,'Precision']:.3f},
accuracy {comp.loc[final_name,'Accuracy']:.3f}; CV F1 {cvtab.loc[final_name,'CV F1 mean']:.3f} ± {cvtab.loc[final_name,'CV F1 std']:.3f}.
{'The gap to the runner-up is below 0.01 F1, i.e. within CV noise.' if close else 'The gap to the runner-up is larger than the CV noise.'}''')

In [ ]:
rf_f1, dt_f1 = comp.loc['Random Forest (tuned)', 'F1'], comp.loc['Decision Tree', 'F1']
best_old = 'Random Forest' if rf_f1 >= dt_f1 else 'Decision Tree'; best_old_f1 = max(rf_f1, dt_f1)
new_best = comp.loc[['Logistic Regression', f'KNN (K={best_k})'], 'F1'].idxmax()
say(f'''### Draft conclusion (edit in your own words)
The best of today's two new models, **{new_best}** (F1 {comp.loc[new_best,'F1']:.3f}), is {'better than' if comp.loc[new_best,'F1'] > best_old_f1 + 0.005 else 'roughly equal to' if abs(comp.loc[new_best,'F1'] - best_old_f1) <= 0.005 else 'worse than'} the best model from Labs 3–4, the {best_old} (F1 {best_old_f1:.3f}).
Feature scaling mattered most for KNN, whose F1 moved from {sc.iloc[1,0]:.3f} (unscaled) to {sc.iloc[1,1]:.3f} (scaled), because distances were otherwise dominated by TotalCharges/MonthlyCharges; for Logistic Regression F1 went from {sc.iloc[0,0]:.3f} to {sc.iloc[0,1]:.3f}, so its main benefits are stable convergence and coefficients that can be compared on one scale, and it would have changed nothing for the trees.
My overall pick is **{final_name}** because it has the best F1/recall balance on the churn class and its cross-validated F1 ({cvtab.loc[final_name,'CV F1 mean']:.3f} ± {cvtab.loc[final_name,'CV F1 std']:.3f}) shows the result is not a one-split fluke.
{'Logistic Regression also gives readable odds ratios, so we learn why customers churn.' if final_name == 'Logistic Regression' else 'Its main drawback is that it is less interpretable than the odds ratios from Logistic Regression.'}
Remaining limitations: all models still miss a sizeable share of churners (best recall {comp['Recall'].max():.2f}) at the default 0.5 threshold, the data is imbalanced, and everything rests on one 80/20 split.
Next I would tune the decision threshold (or use class weights) to raise recall, engineer features such as tenure buckets or number of add-on services, and use repeated cross-validation to put confidence intervals on the comparison.''')

### Answers — Part 7
**Final selection (2–3 sentences).** Take the model named in the output above and justify it with the comparison table: it has the best F1 (and the recall/precision balance that matters for the imbalanced churn problem), and its CV F1 ± std confirms that the lead is stable rather than the luck of one test split — the Lab 4 lesson. If the lead over the runner-up is smaller than the CV standard deviation, say explicitly that the models are effectively tied and that you break the tie using interpretability (Logistic Regression's odds ratios) and training/prediction cost (KNN must store all training data and compute distances at prediction time).

**Conclusion.** Use the auto-generated draft above as the skeleton of your 5–7 sentence conclusion and make sure it answers all four questions: (1) how today's best compares to Labs 3–4's best, (2) what scaling changed, (3) the remaining limitations, (4) what you would try next.